In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data.csv"
panel_data = pd.read_csv(panel_data_path)

In [2]:
panel_data['CovidAffectedPrediction'] = (
    panel_data['Year'].isin([2019, 2020, 2021])
).astype(int)

### Train/validate/test split
Train/validate/test split must be time-aware. Entries in the train set should predate entries in the validation set which should predate entries in the test set. The dataframe year_counts is used to decide which years should be used for the train/validation/test sets.

In [3]:
year_counts = panel_data.groupby('Year').size().reset_index(name = 'NumberOfRows')
year_counts = year_counts.sort_values('Year')
year_counts['Cumulative'] = year_counts['NumberOfRows'].cumsum()
total_rows = year_counts.loc[len(year_counts)-1, 'Cumulative']
year_counts['EntriesPrecedingAndIncluding(%)'] = (year_counts['Cumulative']/total_rows) *100
year_counts

,Year,NumberOfRows,Cumulative,EntriesPrecedingAndIncluding(%)
0,2015,20521,20521,6.723501
1,2016,26591,47112,15.435778
2,2017,31939,79051,25.900273
3,2018,37641,116692,38.232972
4,2019,40916,157608,51.638692
5,2020,17673,175281,57.429074
6,2021,25873,201154,65.906105
7,2022,27153,228307,74.802515
8,2023,35947,264254,86.580192
9,2024,40959,305213,100.000000


- 2022 and before used as train set
- 2023 used for validation set
- 2024 used for test set

In [4]:
panel = panel_data.drop(columns = ['Name', 'WeightClassKg', 'Sex'])
train = panel[panel['Year']<=2022].copy()
validate = panel[panel['Year']==2023].copy()
test = panel[panel['Year']>2023].copy()

train_X = train.drop(columns = 'CompetesNextYear')
train_y = train['CompetesNextYear']
test_X = test.drop(columns = 'CompetesNextYear')
test_y = test['CompetesNextYear']

In [5]:
#checking how balanced the dataset is
numbers = panel_data.groupby('CompetesNextYear').size().reset_index(name = 'Number')
proportions = numbers.copy()
proportions['Proportion'] = numbers['Number']/numbers['Number'].sum()
proportions

,CompetesNextYear,Number,Proportion
0,False,180574,0.591633
1,True,124639,0.408367


### Initial coarse feature selection using random forest

In [6]:
clf_feat_selection = RandomForestClassifier(random_state=42)
clf_feat_selection.fit(train_X, train_y)


importances = clf_feat_selection.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': train_X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False, ignore_index =True)
feature_importance_df


,Feature,Importance
0,TimeSinceLastPBYearEnd,7.487336e-02
1,Age,6.896033e-02
2,Wilks,6.394170e-02
3,Goodlift,6.390510e-02
4,Dots,6.289823e-02
5,TimeCompetingYearEnd,6.250162e-02
6,BestMeetOfYear,5.965354e-02
7,FederationProportion,5.911321e-02
8,PercentageImprovementGradientWithinYear,4.805658e-02
9,ImprovementGradientWithinYear,4.752660e-02


Have chosen HistGradientBoostClassifier (outperformed LightGBM)
Will compare reduced feature set (features greater than e-03) to full feature set using 

In [8]:
important_feats = feature_importance_df.loc[feature_importance_df['Importance']>0.001]
features = important_feats['Feature'].to_list()
columns = features + ['CompetesNextYear']

train_red = train[columns]
train_red_X = train.drop(columns = 'CompetesNextYear')
train_red_y = train['CompetesNextYear']

test_red = test[columns]
test_red_X = test.drop(columns = 'CompetesNextYear')
test_red_y = test['CompetesNextYear']

In [15]:
clf_red = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_red.fit(train_red_X, train_red_y)

preds_red = clf_red.predict(test_red_X)
acc_red = accuracy_score(test_red_y,preds_red)
precision_red = precision_score(test_red_y, preds_red)
recall_red = recall_score(test_red_y, preds_red)
f1_red = f1_score(test_red_y, preds_red)

In [16]:
clf = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf.fit(train_X, train_y)

preds = clf.predict(test_X)
acc = accuracy_score(test_y,preds)
precision = precision_score(test_y, preds)
recall = recall_score(test_y, preds)
f1 = f1_score(test_y, preds)

In [17]:
print('full feature set')
print({'acc': acc, 'precision': precision, 'recall': recall, 'f1': f1})

print('reduced feature set')
print({'acc': acc_red, 'precision': precision_red, 'recall': recall_red, 'f1': f1_red})

full feature set
{'acc': 0.6542151908005567, 'precision': 0.6398597845966267, 'recall': 0.64034775535106, 'f1': 0.6401036769750718}
reduced feature set
{'acc': 0.6542151908005567, 'precision': 0.6398597845966267, 'recall': 0.64034775535106, 'f1': 0.6401036769750718}


Reducing features as above does not impact model performance

Noting that percentage-based features outperform their non-percentage-baseed counterparts, will drop non-percentgae-based features then reevaluate performance. 

In [ ]:
train_perc_X = train_red_X.drop(columns = ['ImprovementGradientWithinYear', 'ImprovementGradientTwoMeets', 
                                           'ImprovementGradientBetweenYears', 'ImprovementAcceleration'])
train_perc_y = train_red_y

test_perc_X = test_red_X.drop(columns = ['ImprovementGradientWithinYear', 'ImprovementGradientTwoMeets', 
                                           'ImprovementGradientBetweenYears', 'ImprovementAcceleration'])
test_perc_y = test_red_y

clf_perc = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_perc.fit(train_perc_X, train_perc_y)

In [22]:
preds_perc = clf_perc.predict(test_perc_X)
acc_perc = accuracy_score(test_perc_y,preds_perc)
precision_perc = precision_score(test_perc_y, preds_perc)
recall_perc = recall_score(test_perc_y, preds_perc)
f1_perc = f1_score(test_perc_y, preds_perc)

In [25]:
print('removing non-percetnatge based performance improvement features')
print({"acc": acc_perc,"precision": precision_perc,"recall": recall_perc,"f1": f1_perc})

removing non-percetnatge based performance improvement features
{'acc': 0.6553626797529236, 'precision': 0.6407033902599706, 'recall': 0.6427881437795516, 'f1': 0.641744073904878}


Tiny increase in performance by removing these features. Will therefore remove these features as they are not necessary for model performance

### Examining correlated features

In [31]:
panel[['Wilks', 'Dots', 'Goodlift']].corr()

,Wilks,Dots,Goodlift
Wilks,1.000000,0.997941,0.992439
Dots,0.997941,1.000000,0.996222
Goodlift,0.992439,0.996222,1.000000


keeping Wilks only

In [ ]:
train_wilks_X = train_perc_X.drop(columns = ['Dots', 'Goodlift'])
train_wilks_y = train_perc_y

test_wilks_X = test_perc_X.drop(columns = ['Dots', 'Goodlift'])
test_wilks_y = test_perc_y

clf_wilks = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_wilks.fit(train_wilks_X, train_wilks_y)


preds_wilks = clf_wilks.predict(test_wilks_X)
acc_wilks = accuracy_score(test_wilks_y,preds_wilks)
precision_wilks = precision_score(test_wilks_y, preds_wilks)
recall_wilks = recall_score(test_wilks_y, preds_wilks)
f1_wilks = f1_score(test_wilks_y, preds_wilks)

In [34]:
print('discarding Goodlift and Dots')
print({"acc": acc_wilks,"precision": precision_wilks,"recall": recall_wilks,"f1": f1_wilks})

removing non-percetnatge based performance improvement features
{'acc': 0.6545814106789717, 'precision': 0.6405478336133598, 'recall': 0.63963597539275, 'f1': 0.6400915797506995}


In [35]:
train_gl_X = train_perc_X.drop(columns = ['Dots', 'Wilks'])
train_gl_y = train_perc_y

test_gl_X = test_perc_X.drop(columns = ['Dots', 'Wilks'])
test_gl_y = test_perc_y

clf_gl = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_gl.fit(train_gl_X, train_gl_y)


preds_gl = clf_gl.predict(test_gl_X)
acc_gl = accuracy_score(test_gl_y,preds_gl)
precision_gl = precision_score(test_gl_y, preds_gl)
recall_gl = recall_score(test_gl_y, preds_gl)
f1_gl = f1_score(test_gl_y, preds_gl)

In [36]:
print('discarding Wilks and Dots')
print({"acc": acc_gl,"precision": precision_gl,"recall": recall_gl,"f1": f1_gl})

discarding Goodlift and Dots
{'acc': 0.6561195341683147, 'precision': 0.6421878182929314, 'recall': 0.6411103767349636, 'f1': 0.641648645210533}


# How many features to keep

Will take top N features where N is determined by taking top n features and seeing which value of n yields best performance on validation set. Can then use top N features to retrain classifier and evaluate performance using test set.

In [ ]:
importances = clf.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': train_X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False, ignore_index =True)

feature_importance_df

In [ ]:
panel[['Wilks','Dots','Goodlift']].corr()


Can also see that percentage-based improvement features perform better than raw improvement. Will drop raw improvement features

In [ ]:
reduced_feats = panel_data.drop(columns = ['Goodlift', 'Dots', 'ImprovementGradientWithinYear','ImprovementGradientBetweenYears', 'ImprovementAcceleration'])
reduced_feats = reduced_feats.loc[reduced_feats['Importance']>0.001]

Wilks, Goodlift and Dots are all different formulas for bodyweight adjusted strength. Very high correlation between these so will only keep most predictive feature, found to be Dots.

In [ ]:
# reduced_feats = feature_importance_df.loc[:13]
# reduced_feats = reduced_feats.drop(labels = [2,4], axis = 0)
# reduced_feats[:4]

In [ ]:
results = []

for i in range(len(feature_importance_df)):
    features = feature_importance_df.loc[:i, 'Feature'].to_list()
    columns = features + ['CompetesNextYear']

    train_n = train[columns]
    train_n_X = train_n.drop(columns = 'CompetesNextYear')
    train_n_y = train_n['CompetesNextYear']

    validate_n = validate[columns]
    validate_n_X = validate_n.drop(columns = 'CompetesNextYear')
    validate_n_y = validate_n['CompetesNextYear']

    clf_n = RandomForestClassifier(random_state=42)
    clf_n.fit(train_n_X, train_n_y)
    
    preds_n = clf_n.predict(validate_n_X)
    acc_n = accuracy_score(validate_n_y, preds_n)
    f1_n = f1_score(validate_n_y, preds_n)
    precision_n = precision_score(validate_n_y, preds_n)
    recall_n = recall_score(validate_n_y, preds_n)

    results.append({'n_features': len(features), 'features': features, 
                    'accuracy': acc_n,'f1': f1_n, 'precision_n': precision_n, 'recall_n':recall_n})

results_df = pd.DataFrame(results)

In [ ]:
results_df